# A Privacy-Preserving Federated Learning Framework for Multi-Class Bengali Fake News Detection on Resource-Constrained Devices

## Abstract

The proliferation of fake news in low-resource languages like Bengali poses a significant challenge to information integrity. Existing detection methods rely on centralized training, which requires aggregating sensitive user data and raises privacy concerns.

In this notebook, we implement a privacy-preserving federated learning framework for 4-class Bengali fake news detection (Fake, True, Mostly True, Mostly Fake). We compare three approaches:

1. **Centralized BanglaBERT** (baseline)
2. **Federated FedAvg**
3. **Federated FedProx** (proposed)

## Dataset
- **BanFakeNews-2.0** (60,544 articles, 4 classes)
- Source: [Kaggle](https://www.kaggle.com/datasets/hrithikmajumdar/bangla-fake-news)

## Key Results

| Method | Accuracy | F1 (macro) |
|---|---|---|
| Centralized BanglaBERT | 76.45% | 0.4923 |
| Federated FedAvg | 77.66% | 0.4749 |
| **Federated FedProx** | **80.95%** | **0.4990** |

In [4]:
# Install required libraries
!pip install -q transformers scikit-learn

## 1. Imports and Configuration

In [5]:
import os, json, random, time, re, shutil, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils import resample

# ============ Configuration ============
SEED = 42
NUM_CLIENTS = 5
ALPHA = 1.0
NUM_ROUNDS = 10
LOCAL_EPOCHS = 2
BATCH_SIZE = 16
MAX_LEN = 128
LR_BERT = 2e-5
LR_HEAD = 1e-3
FEDPROX_MU = 0.01
BANGLA_MODEL = "csebuetnlp/banglabert"

# Paths
KAGGLE_BASE = "/kaggle/input/datasets/hossmurad"
PROJECT_ROOT = "/kaggle/working"
DATA_DIR = f"{PROJECT_ROOT}/data"

for d in [DATA_DIR, f"{PROJECT_ROOT}/models", f"{PROJECT_ROOT}/results", f"{PROJECT_ROOT}/plots"]:
    os.makedirs(d, exist_ok=True)

# Reproducibility
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

 Device: cuda
   GPU: Tesla T4


## 2. Data Loading

Load the BanFakeNews-2.0 dataset from Kaggle input.

In [8]:
# Copy CSV files from Kaggle input to working directory
for src_name, dst_name in [
    ("fake-news-train-cleaned-data/train_cleaned.csv", "train_cleaned.csv"),
    ("val-cleaned-data/val_cleaned.csv", "val_cleaned.csv"),
    ("bangla-fake-news-test-cleaned-data/test_cleaned.csv", "test_cleaned.csv"),
]:
    src = f"{KAGGLE_BASE}/{src_name}"
    dst = f"{DATA_DIR}/{dst_name}"
    if not os.path.exists(dst):
        shutil.copy(src, dst)

# Load datasets
train_raw = pd.read_csv(f"{DATA_DIR}/train_cleaned.csv")
val_df = pd.read_csv(f"{DATA_DIR}/val_cleaned.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test_cleaned.csv")

print(f" Dataset loaded:")
print(f"   Train: {train_raw.shape}")
print(f"   Val:   {val_df.shape}")
print(f"   Test:  {test_df.shape}")

label_names = {0: "Fake", 1: "True", 2: "Mostly True", 3: "Mostly Fake"}
print(f"\nClass distribution (train):")
for lbl, cnt in train_raw["Label"].value_counts().sort_index().items():
    print(f"   {lbl} ({label_names[lbl]}): {cnt:,} ({cnt/len(train_raw)*100:.2f}%)")

 Dataset loaded:
   Train: (42380, 3)
   Val:   (9082, 3)
   Test:  (9082, 3)

Class distribution (train):
   0 (Fake): 1,524 (3.60%)
   1 (True): 2,112 (4.98%)
   2 (Mostly True): 4,629 (10.92%)
   3 (Mostly Fake): 34,115 (80.50%)


## 3. Data Preprocessing

### 3.1 Text Cleaning
Combine Headline + Content, remove URLs and HTML tags.

### 3.2 Class Balancing
Balance training set to 3,000 samples per class.

In [9]:
def make_text(row):
    """Combine Headline and Content with [SEP] separator."""
    h = str(row["Headline"]) if pd.notna(row["Headline"]) else ""
    c = str(row["Content"]) if pd.notna(row["Content"]) else ""
    return (h + " [SEP] " + c).strip()

def clean_bengali_text(text):
    """Basic cleaning for Bengali news text."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\u0980-\u09FF\s\.,!?;:\-()0-9\[\]SEP]", " ", text)
    return text.strip()

# Apply cleaning
for d in [train_raw, val_df, test_df]:
    d["text"] = d.apply(make_text, axis=1)
    d["clean_text"] = d["text"].apply(clean_bengali_text)

train_raw = train_raw.dropna(subset=["clean_text", "Label"]).reset_index(drop=True)
val_df = val_df.dropna(subset=["clean_text", "Label"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["clean_text", "Label"]).reset_index(drop=True)

# Balance training set
TARGET_PER_CLASS = 3000
balanced_dfs = []
for cls in sorted(train_raw["Label"].unique()):
    cls_df = train_raw[train_raw["Label"] == cls]
    if len(cls_df) >= TARGET_PER_CLASS:
        sampled = cls_df.sample(n=TARGET_PER_CLASS, random_state=SEED)
    else:
        sampled = resample(cls_df, replace=True, n_samples=TARGET_PER_CLASS, random_state=SEED)
    balanced_dfs.append(sampled)

train_balanced = pd.concat(balanced_dfs, ignore_index=True)
train_balanced = train_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f" Balanced train: {len(train_balanced):,} samples (3,000/class)")
print(f"   Val:  {len(val_df):,}")
print(f"   Test: {len(test_df):,}")

 Balanced train: 12,000 samples (3,000/class)
   Val:  9,082
   Test: 9,082


## 4. Federated Data Partitioning

Partition training data into 5 clients using Dirichlet distribution (α=1.0) to simulate Non-IID conditions.

In [10]:
def dirichlet_split(labels, num_clients, alpha=1.0, seed=42):
    """Partition data into num_clients using Dirichlet distribution (Non-IID)."""
    np.random.seed(seed)
    labels = np.array(labels)
    num_classes = len(np.unique(labels))
    client_indices = [[] for _ in range(num_clients)]

    for c in range(num_classes):
        class_idx = np.where(labels == c)[0]
        np.random.shuffle(class_idx)
        proportions = np.random.dirichlet([alpha] * num_clients)
        proportions = (np.cumsum(proportions) * len(class_idx)).astype(int)[:-1]
        splits = np.split(class_idx, proportions)
        for cid, split in enumerate(splits):
            client_indices[cid].extend(split.tolist())

    for i in range(num_clients):
        np.random.shuffle(client_indices[i])
    return client_indices

client_indices = dirichlet_split(train_balanced["Label"].values, NUM_CLIENTS, ALPHA, SEED)

print(f" Federated split complete (α={ALPHA})")
print(f"{'Client':<8} {'Total':<8} {'Fake':<6} {'True':<6} {'M.True':<8} {'M.Fake':<8}")
print("-" * 55)
for i, idx in enumerate(client_indices):
    dist = train_balanced.iloc[idx]["Label"].value_counts().sort_index().to_dict()
    print(f"Client {i:<3} {len(idx):<8} "
          f"{dist.get(0,0):<6} {dist.get(1,0):<6} {dist.get(2,0):<8} {dist.get(3,0):<8}")

 Federated split complete (α=1.0)
Client   Total    Fake   True   M.True   M.Fake  
-------------------------------------------------------
Client 0   2998     759    69     209      1961    
Client 1   1754     409    1281   44       20      
Client 2   2833     765    310    1552     206     
Client 3   2011     219    1181   289      322     
Client 4   2404     848    159    906      491     


## 5. Dataset and Model Architecture

### 5.1 BengaliFakeNewsDataset
Custom PyTorch Dataset for tokenizing Bengali text.

### 5.2 BengaliFakeNewsClassifier
BanglaBERT (ELECTRA-based) with top-4 encoder layers unfrozen + linear classification head.

In [11]:
class BengaliFakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }


class BengaliFakeNewsClassifier(nn.Module):
    def __init__(self, model_name=BANGLA_MODEL, num_classes=4,
                 dropout=0.3, unfreeze_top_n=4):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        for p in self.bert.parameters():
            p.requires_grad = False

        total_layers = len(self.bert.encoder.layer)
        for layer_idx in range(total_layers - unfreeze_top_n, total_layers):
            for p in self.bert.encoder.layer[layer_idx].parameters():
                p.requires_grad = True

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))


# Load tokenizer
print("Loading BanglaBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BANGLA_MODEL)
print(f" Tokenizer loaded (vocab size: {tokenizer.vocab_size:,})")

Loading BanglaBERT tokenizer...


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

 Tokenizer loaded (vocab size: 32,000)


## 6. DataLoaders

Create PyTorch DataLoaders for centralized and federated training.

In [12]:
train_ds = BengaliFakeNewsDataset(
    train_balanced["clean_text"].values, train_balanced["Label"].values, tokenizer, MAX_LEN
)
val_ds = BengaliFakeNewsDataset(
    val_df["clean_text"].values, val_df["Label"].values, tokenizer, MAX_LEN
)
test_ds = BengaliFakeNewsDataset(
    test_df["clean_text"].values, test_df["Label"].values, tokenizer, MAX_LEN
)

centralized_loaders = {
    "train": DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
    "val":   DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
    "test":  DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
}

client_loaders = []
for i, idx in enumerate(client_indices):
    sub_df = train_balanced.iloc[idx].reset_index(drop=True)
    ds = BengaliFakeNewsDataset(
        sub_df["clean_text"].values, sub_df["Label"].values, tokenizer, MAX_LEN
    )
    client_loaders.append(
        DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    )

print(f" DataLoaders ready:")
print(f"   Centralized — Train: {len(train_ds):,}, Val: {len(val_ds):,}, Test: {len(test_ds):,}")
for i, l in enumerate(client_loaders):
    print(f"   Client {i}: {len(l.dataset):,} samples")

 DataLoaders ready:
   Centralized — Train: 12,000, Val: 9,082, Test: 9,082
   Client 0: 2,998 samples
   Client 1: 1,754 samples
   Client 2: 2,833 samples
   Client 3: 2,011 samples
   Client 4: 2,404 samples


## 7. Training and Evaluation Functions

In [13]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate model on a data loader."""
    model.eval()
    total_loss, total_samples = 0.0, 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        total_samples += labels.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    return {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(all_labels, all_preds),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(all_labels, all_preds, average="weighted", zero_division=0),
    }


def train_one_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


def get_model_weights(model):
    """Extract weights to CPU for federated communication."""
    return {k: v.cpu().clone() for k, v in model.state_dict().items()}


def set_model_weights(model, weights):
    """Load weights into model."""
    model.load_state_dict(weights)


def federated_average(global_model, client_weights_list, client_sizes):
    """FedAvg: weighted average by sample count."""
    total_samples = sum(client_sizes)
    new_weights = {}
    for key in global_model.state_dict().keys():
        new_weights[key] = torch.zeros_like(
            global_model.state_dict()[key], dtype=torch.float32, device="cpu"
        )
    for cw, n_i in zip(client_weights_list, client_sizes):
        wf = n_i / total_samples
        for key in new_weights.keys():
            new_weights[key] += cw[key].float() * wf
    return new_weights


print(" Training functions defined")

 Training functions defined


## 8. Centralized BanglaBERT (Baseline)

In [ ]:
print("=" * 70)
print("CENTRALIZED BANGLABERT TRAINING")
print("=" * 70)

model = BengaliFakeNewsClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([
    {"params": model.bert.parameters(), "lr": LR_BERT},
    {"params": model.classifier.parameters(), "lr": LR_HEAD}
], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

history = {"epoch": [], "val_f1": [], "val_acc": []}
best_f1 = 0.0

for epoch in range(1, 6):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(
        model, centralized_loaders["train"], optimizer, criterion, device
    )
    val_metrics = evaluate(model, centralized_loaders["val"], criterion, device)
    scheduler.step()
    elapsed = time.time() - t0

    history["epoch"].append(epoch)
    history["val_f1"].append(val_metrics["f1_macro"])
    history["val_acc"].append(val_metrics["accuracy"])

    print(f"Epoch {epoch}/5 | train_loss={train_loss:.4f} acc={train_acc:.4f} | "
          f"val_acc={val_metrics['accuracy']:.4f} f1={val_metrics['f1_macro']:.4f} | {elapsed:.1f}s")

    if val_metrics["f1_macro"] > best_f1:
        best_f1 = val_metrics["f1_macro"]
        torch.save(model.state_dict(), f"{PROJECT_ROOT}/models/centralized_best.pt")

print(f"\n Best centralized F1: {best_f1:.4f}")

del model, optimizer, scheduler
gc.collect()
torch.cuda.empty_cache()

CENTRALIZED BANGLABERT TRAINING


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/5 | train_loss=1.0981 acc=0.5067 | val_acc=0.8268 f1=0.4594 | 200.6s


## 9. Federated FedAvg

In [ ]:
def train_client(model, loader, criterion, device, local_epochs=2):
    """Local training on a single client."""
    model.train()
    optimizer = torch.optim.AdamW([
        {"params": model.bert.parameters(), "lr": LR_BERT},
        {"params": model.classifier.parameters(), "lr": LR_HEAD}
    ], weight_decay=0.01)

    total_loss, total_correct, total_samples = 0.0, 0, 0
    for _ in range(local_epochs):
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
    return total_loss / total_samples, total_correct / total_samples


print("=" * 70)
print("FEDERATED LEARNING — FedAvg")
print("=" * 70)

global_model = BengaliFakeNewsClassifier().to(device)
criterion = nn.CrossEntropyLoss()
client_sizes = [len(l.dataset) for l in client_loaders]

history_avg = {"round": [], "val_f1": [], "val_acc": []}
best_f1 = 0.0

for round_idx in range(1, NUM_ROUNDS + 1):
    t0 = time.time()
    global_weights = get_model_weights(global_model)
    client_weights_list = []

    for cid in range(NUM_CLIENTS):
        cm = BengaliFakeNewsClassifier().to(device)
        set_model_weights(cm, global_weights)
        cm.to(device)
        train_client(cm, client_loaders[cid], criterion, device, LOCAL_EPOCHS)
        client_weights_list.append(get_model_weights(cm))
        del cm
        torch.cuda.empty_cache()

    new_weights = federated_average(global_model, client_weights_list, client_sizes)
    set_model_weights(global_model, new_weights)
    global_model.to(device)

    val_metrics = evaluate(global_model, centralized_loaders["val"], criterion, device)
    elapsed = time.time() - t0

    history_avg["round"].append(round_idx)
    history_avg["val_f1"].append(val_metrics["f1_macro"])
    history_avg["val_acc"].append(val_metrics["accuracy"])

    print(f"Round {round_idx}/{NUM_ROUNDS} | val_acc={val_metrics['accuracy']:.4f} "
          f"f1={val_metrics['f1_macro']:.4f} | {elapsed:.1f}s")

    if val_metrics["f1_macro"] > best_f1:
        best_f1 = val_metrics["f1_macro"]
        torch.save(global_model.state_dict(), f"{PROJECT_ROOT}/models/federated_fedavg_best.pt")

print(f"\n Best FedAvg F1: {best_f1:.4f}")

del global_model, criterion
gc.collect()
torch.cuda.empty_cache()

## 10. Federated FedProx (Proposed)

FedProx adds a proximal term to the local loss:
$$\mathcal{L} = \mathcal{L}_{CE} + \frac{\mu}{2} \|w_{local} - w_{global}\|^2$$

In [ ]:
def train_client_fedprox(model, global_params, loader, criterion, device,
                          mu=0.01, local_epochs=2):
    """FedProx local training with proximal term."""
    model.train()
    optimizer = torch.optim.AdamW([
        {"params": model.bert.parameters(), "lr": LR_BERT},
        {"params": model.classifier.parameters(), "lr": LR_HEAD}
    ], weight_decay=0.01)

    total_loss, total_correct, total_samples = 0.0, 0, 0
    for _ in range(local_epochs):
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            ce_loss = criterion(logits, labels)

            prox = 0.0
            for n, p in model.named_parameters():
                if p.requires_grad:
                    prox += ((p - global_params[n].to(p.device)) ** 2).sum()
            loss = ce_loss + (mu / 2.0) * prox

            loss.backward()
            optimizer.step()
            total_loss += ce_loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
    return total_loss / total_samples, total_correct / total_samples


print("=" * 70)
print("FEDERATED LEARNING — FedProx (Proposed)")
print("=" * 70)

global_model = BengaliFakeNewsClassifier().to(device)
criterion = nn.CrossEntropyLoss()

history_prox = {"round": [], "val_f1": [], "val_acc": []}
best_f1 = 0.0

for round_idx in range(1, NUM_ROUNDS + 1):
    t0 = time.time()
    global_weights = get_model_weights(global_model)
    client_weights_list = []

    for cid in range(NUM_CLIENTS):
        cm = BengaliFakeNewsClassifier().to(device)
        set_model_weights(cm, global_weights)
        cm.to(device)
        train_client_fedprox(cm, global_weights, client_loaders[cid], criterion, device,
                              FEDPROX_MU, LOCAL_EPOCHS)
        client_weights_list.append(get_model_weights(cm))
        del cm
        torch.cuda.empty_cache()

    new_weights = federated_average(global_model, client_weights_list, client_sizes)
    set_model_weights(global_model, new_weights)
    global_model.to(device)

    val_metrics = evaluate(global_model, centralized_loaders["val"], criterion, device)
    elapsed = time.time() - t0

    history_prox["round"].append(round_idx)
    history_prox["val_f1"].append(val_metrics["f1_macro"])
    history_prox["val_acc"].append(val_metrics["accuracy"])

    print(f"Round {round_idx}/{NUM_ROUNDS} | val_acc={val_metrics['accuracy']:.4f} "
          f"f1={val_metrics['f1_macro']:.4f} | {elapsed:.1f}s")

    if val_metrics["f1_macro"] > best_f1:
        best_f1 = val_metrics["f1_macro"]
        torch.save(global_model.state_dict(), f"{PROJECT_ROOT}/models/federated_fedprox_best.pt")

print(f"\n Best FedProx F1: {best_f1:.4f}")

del global_model, criterion
gc.collect()
torch.cuda.empty_cache()

## 11. Final Test Evaluation

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    preds_all, labels_all = [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        logits = model(input_ids, attention_mask)
        preds_all.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        labels_all.extend(labels.cpu().numpy().tolist())
    return np.array(labels_all), np.array(preds_all)


results = {}
for name, ckpt in [
    ("Centralized", "centralized_best.pt"),
    ("FedAvg", "federated_fedavg_best.pt"),
    ("FedProx", "federated_fedprox_best.pt"),
]:
    model = BengaliFakeNewsClassifier().to(device)
    model.load_state_dict(torch.load(f"{PROJECT_ROOT}/models/{ckpt}", map_location=device))
    model.eval()

    y_true, y_pred = predict(model, centralized_loaders["test"], device)
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1w = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    results[name] = {
        "accuracy": acc, "f1_macro": f1m, "f1_weighted": f1w,
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }

    print(f"{name:12s} | Acc={acc:.4f} | F1-macro={f1m:.4f} | F1-weighted={f1w:.4f}")

    del model
    torch.cuda.empty_cache()

with open(f"{PROJECT_ROOT}/results/final_comparison.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n Results saved")

## 12. Publication Figures

In [ ]:
import matplotlib.pyplot as plt

# Figure 1: Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_avg["round"], history_avg["val_f1"], "o-",
             label="FedAvg", color="#3498db", linewidth=2, markersize=8)
axes[0].plot(history_prox["round"], history_prox["val_f1"], "s-",
             label="FedProx", color="#e74c3c", linewidth=2, markersize=8)
axes[0].axhline(y=results["Centralized"]["f1_macro"], color="green",
                linestyle="--", label=f"Centralized ({results['Centralized']['f1_macro']:.4f})")
axes[0].set_xlabel("Round", fontsize=12)
axes[0].set_ylabel("Validation F1 (macro)", fontsize=12)
axes[0].set_title("(a) F1 Progression", fontsize=13)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history_avg["round"], history_avg["val_acc"], "o-",
             label="FedAvg", color="#3498db", linewidth=2, markersize=8)
axes[1].plot(history_prox["round"], history_prox["val_acc"], "s-",
             label="FedProx", color="#e74c3c", linewidth=2, markersize=8)
axes[1].axhline(y=results["Centralized"]["accuracy"], color="green",
                linestyle="--", label=f"Centralized ({results['Centralized']['accuracy']:.4f})")
axes[1].set_xlabel("Round", fontsize=12)
axes[1].set_ylabel("Validation Accuracy", fontsize=12)
axes[1].set_title("(b) Accuracy Progression", fontsize=13)
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Federated Learning Dynamics (5 Non-IID Clients, α=1.0)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{PROJECT_ROOT}/plots/fig1_training_curves.png", dpi=200, bbox_inches="tight")
plt.show()

# Figure 2: Method comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
methods = list(results.keys())
colors = ["#27ae60", "#3498db", "#e74c3c"]

for ax, metric, ylabel in [
    (axes[0], "accuracy", "Test Accuracy"),
    (axes[1], "f1_macro", "F1 (macro)"),
    (axes[2], "f1_weighted", "F1 (weighted)"),
]:
    values = [results[m][metric] for m in methods]
    bars = ax.bar(methods, values, color=colors, edgecolor="black", linewidth=1.5)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.4f}",
                ha="center", fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(ylabel, fontsize=13)
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("Centralized vs Federated BanglaBERT", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{PROJECT_ROOT}/plots/fig2_method_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"\n Figures saved")
print(f"\n📊 FINAL RESULTS:")
for m in methods:
    print(f"   {m:12s}: Acc={results[m]['accuracy']:.4f} | F1-macro={results[m]['f1_macro']:.4f}")